<a href="https://colab.research.google.com/github/ashok0432/mri/blob/main/longnet_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dual-Stream Longitudinal Network – Colab runner
M.Tech project (Ashok Raj A). Runs the `longnet` code base end to end.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is enough for 96³ with `--base 16 --bs 2 --amp`).

Order: 0 setup → 1 smoke test → 2 data → 3 preprocess → 4 train + ablations → 5 results table → 6 attention maps.

## 0. Setup

In [20]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/mtech_longnet'   # everything persistent lives here
os.makedirs(WORK, exist_ok=True)

GPU 0: Tesla T4 (UUID: GPU-8bb5b7f6-e60b-2b9b-a7a5-3234ef41f97b)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
# Project code is embedded in this notebook - no upload needed.
import os, io, base64, tarfile, shutil, subprocess, sys
B64 = "H4sIAAAAAAAAA+w8/XPbNrL52X8F6pt7BROKESnZbtTq5tI4aTOXuB7b9zodjZ6GIkGJJ36VgGzJfb2//e0uwA9RcpK2ad68eeHEEgUsFsDuYr8AJMmzRSbU00d/4tOH5+zkhL7h6X7Tu3vinXrD/uDsdPio77reYPCInfyZg6qetVR+ydijMs/Vu+DeV/9/9EkM/5M/UQ4+nP/uyekQ+e+d9d3P/P8UT5f/aR6KxCm2H7MPZDDx9SD/3f7p2Znmv+fCcwr8Hw5OvEes/zEH8dDz/5z/x8fH52s/6V2rUvgpewNiEKt1GGd+wi6EusvLlXPEekzdxRnjeaHiHKqSLbsT8WKpenLplyK02OCciSwA4Skly29FydRSQKOcqTgVRR5nSiKaoMyl7GEZ85USGaKDN4Ke50olIhPBinHhB8umKft5LcpYSALL4aO0EFkoqEPd3yRybRZ5+NeL3Cm7i9WShXEUiRIGJphcxQUL8gzwY6eS9f7GbvONSABNonyW+gXiXCT5HKa+FH6IEFfPL35gi9IPBeMvrp5eXj29Pn96eW4d4ToZsS+TFr2+ZDwUkb9OlMX+m30p42yRCCic+1IkcQbgeQaEwzlEeZLkd711wWTgZyyWTAqRWcCMozgt8hIIkpfB0tZfTgY0kizLmt9OtM4CzQusenV0dAR9s3mSByse2yy3mRy71uiIwVMKtS4zaO9cC6AkEN1POPx6kWe3g9CAD6CFzYCG89iX41d+IoVlY5vvynxdXORlytM4418BsKX/sO7lm39yy6ZeDj5NL3nVi/s7e7FgjkHiS8leakHDKbzNw3UizDyRArNZnMVqNuNA88hmcTYLluMh9AdMGLunBhIfuS4Ah+XULay6CmRvzCbYRDdkj5lXvw3rt6+mDTLozZF9aGZYgP3agGjSn1pfm2q3rqZyqnbhy7M6eLw2oKsBPQKsUA3aEJ6GGBxARTMJlkc1faK8vPPL0JBn06JHhKM38+Ab6Cly6wKXR30s8eoSj0culgzqkgGPWr0bmZtEfViR1cocTGsevkBFcAML/HmlBvbYCavh2ziMS1EJulYejd6Yg36CZcOihMpE2FYhKl8JWOV5hIUtXSIdXGUPiEsYpzatfYkyk/qbmUYzHvaffaDsEDF8FymFEwJtECPCZppNHyhIKljOoriUanxTrkUXjed+DDQZ0F+/mFG98beipMUGWPQa2y3abV/kUre79Es/FQqmrvXQvQCOcLdNKaKhBZIB4EgbR5WgqWYZYPYTTWbEBwNS4djpd+U1isyMdzXVoQGDRvVLTQYPFmNdYXRSC8ZUm6F1O/Q+bYcg/WoG8ppBtxd5Jh5amgsg6wIWzUqIguCNtmyE8FubvbDZuc2+t9mPgG3hOmCNC1EDKFeXmvXBPQu44WcS6C+Qa6hOFE5/4b0DpkZXVEsd6iYjm42U6RBUVKMJf4ZGP9d6InO5ci32hBW1CHLlUUHdIocWd7WugaXDNQ79lwkRzrSzIcc1MWzmg9H3F4J+1fUk/Iz9BadO8wnBZwAfxGs6A5Szui/P5dSPq/92+tLkrhvGUcOK0T4r71wnFAq8Fk79T4A5F4DxwpsiX5nYFIkPTsI8TmK1ZfxyiSakRVziFXw8AXIYrsAH/PIOgFRLBWnbBq4kGklcN1vj8BI/nYc+UyOmuvx1SkFc5DvytKfJ1xn2ZtO319jifxYfZIZt8r/Qor9LiRrrBl7abWNJcV6m7YNmTKNv4d5A+1eggEBXFTmItuAIE9+LMQIase0NRmAy0ZMbH6syTmjxHoNoJfEC7HdeZuDNduXAkKMeqFGFga/4xIwDkLotdwX9a+1eg0P9e5wWEMwZOanjAb4C+3KsrezUvvdFTvnOorC18dSKRBfoebcd2OPfYOGwtVnSDeZZjrKmq5rS3YbgjKMcVz6ccZRwptYeYK1GqBGswJ2JMQGMeTcicn9qDOANdeyrXrsHfBHjThGJLey5mY7ullR3hewvLFpLcD8i4YNwYGjDKIAB5z4R6Eo2sQmEGBCddGwQNB5oK2ScZeoejMigduzcDnXWBRIHVp+pJzewbuHVTue6cGs4r3I5azh3uoe2X4Mbx7PfgPe74CSVMwqWOsNH99ZI7f7gUYJbrTqmV9vQigRANdf7as/Snpd5ka8V7zvejukF2GqNWEZf/B0ifBUH4Lgs87BZc7DiOYxtbo2qNd1ayVgB/4Bbvib+Q5oHGLt5p5VGqa0WDBuP2bGOC49HKDZNaJjlzA8CIbW9ghCxKGOwG7XburPIN2QJGhdslsQr0HAtpa8Fri38fIPau15ZO9Da1wDoyCVhijz42psBCT+EqlmuSPxHTbu6nu+5LU0v9xUgybt2BYkLuk3LUQorSJByfm9GTaAwQE8PEET8YAOXh50Grm7gPtCgv9egrxv02w2KPE9EWJOdhIQvcDkvXMvx5xJ0ZSr8jHOPotyhBdS+7xbtLmQjdmEcKK71e2dR8RCQkLrfXTZcjwaG94fzP938n9xmIH2wXD5iDvDd+b/+0K3yv57r9k+hHH4MP+f/PskDwfB1xXLWdgbY26vXrPAhpiQHVqYQ3vWUkArUl1ZQcUG6izJtoIrZzYvXz40Sc46eg3T7SS/0lc8gAmQp0JnBumdYArKFEUApijIncPgdZ1KBaDutHFi2TqECM1/FUVTmKQOPew7Wl5l6vVBSv9CLZIYws9s8WadCmoTYDJzIOSd3DywYGJlSIHAYr6XR0fegX7Zb0OLoMmaFky/KOJyMtIeImmBUxTj1q2fCHVQHnN/fY16TUKPKYI/BZsHs+HbblLutcuioLvdMudXO04Xsm7EZI1WaqaT+SsyQIXo6Yz4ES6f/UEtkizEqZVBlgNsH40E/qwwg8GxMn8BKmCSQK8xTx6QrZ1Bh/LuqMarH6hX1f/3eqH/tB4Fp4tCcXO0FuMu8T4oOjduLK/vyyr4+ty/PCXegCeyXpb/Vc7DYU6LJDoIe+LtnoDANTVw9cGedxRHG4yc2e2aq0PL80h+xvgOduiOEfbwD23cGAN53ToFA3sH6Z+heOP2TnRzm4ACo62DqEjD9OqmIoYVgXkJcBwPZkbQD07RZJVKAue8MDc9Tt9s2AGZCyPwfGrMGwpnyPSivhiIXFejxN9bXbIEB6OSMAZ+DwTAcXsaRMjRtaI6RINIc+QbxYBxCzMLSWJZiEUsIGNEppuawvDAZNNGTR82QMtwZSDEL5LX8nlvNbT0KPrQx5m+oEyW5rwZthwUwrRATiOZC8GELE1EZw+WxIXaXiUPk8VfYAU/3Kk+RbUCpx4xTCLFiX4xB6IhKvb5z0rLxNOzJaoodYX9aMHXqCuUa5OzEsLFphARx/KIQWchvrZpG6GUTAco8SbgpsZlaF4ngxAOQSV6R3ar1Sa3TKopqUQf1hjzrajkAcuQ6RZ849fTbjiqhflGJ1QMIta4EeNfxpdpC3N9iB6HZL65D2dpUXKJheDCAzcanQ0OojpqSQoTjfou5WjMdVkkI3IkcYiVSkr+ONiQF2NJ9MfsrqCGSqriRqsyaHrXGnIjMDLmJAaCMNz1ZbfAFTBwKmzC97eW3dm3qQh1EIMWR3LZexg3ySTw96Aoqd6ydTOI0GUHuA+2Ut18+h3LtOu5VhQ/sy5A/qaEh2pV5yUvkuy9XB/pNXVN3oO/U+xgO6Ofnf/Xp+v/VSv6YRwDe4/835z9q/394dvrZ//8kT5f/O075R+rjPfw/GZydduO/4eDkM/8/xQPh1mXNcgzt0GRi6MYuXkc3rylcA+MYL+KQBXmv7Q8yfg1mLxGvb/4BRoJcs958nm8wV1rYR/c9GeQluKLkP8WS2tjkXd4Lh12v1knSA1xxUUC/I4oSvz/vffvyhs0FjEIswSNg/NvSv7nGuMNPYFjhlukm6BlgfYFuSa8nEVtVY2OUI9dFkWyNx4gWTD4UWlZF8RyjSyqM5zrglEFcbJ0sjFN/ISojf5/nqYnJwD8KeeGrZeekBbj/UgcAgMppoBzwIWYR0hRdthD9rPGun4VYidozTWpR8ijeiHCG7TFpfwvEMj8g6KY3E/jhJPd+A0w7EAQSXBEv+WnvPH9lsR2G5pHpAI++ULf6GE26VhgPxBn61ATqsCuaKrAFp+kQ8pulYNJPBaMdJoQlvgEfYsBk8pqmA7VO83VJY2QcM7dCAgswsT+HcopnFvGtyOrdekP8WuSQTzJWKx2HYPCOv5wrkJHXyK0dslEVfrxqxx3p7X6rHQIfboYGsmr4QuAWkwhvqhm/hsoYxP0eGbcBJtwaNC/XiSgH5zXgO4/NHHre29+rOIE657uXP7x9eXP1kx7sVTVSmt5Vi9tvKR9u4v4r51ooKCnj4Ll8i1un8i0x/XXDc37S3wO+9oEdQLBrRCoWW37lXD2/OP/hrfX1IbBLUWLmAwndd8AjrqJgAn1d79aBT1qTXuf1W2A/FCpOcb7PJUxnnfjltRLFd5gwAdTnAmMABfE65gRED+IQt98/1PwaN2fkK1jml8utjOHXNYZlbYJcL8s4W73yAxiRhLG/Ebci4ZMhRW3utJrjdQq2YYlUiBcg0A2gR0edql0TM0niVyMGKE245XeZ+IFobzYqlOkr5+VGBGsljDBV0Z1sJFcibUFwQdAQRm1acquJR5Grkd2o1hojDfZjCcGIln1A2yiVGr5WKwz1baVUmtAnPbCIqiat6K3TV2fwB8Z+oZXChdYJOAmr0Wk7US41+U6o56iKkKH1dDBq2hitek/mCENxowsz3JCAnxP4w8zJtJpxdu+gjWLfgOiMWqE01WOmc8wQDeshJGX4LUzxcGynYEWxJyh5p1YFTx2Mx9ADtOy3Rw6V1ck9tFMzNJ+ULMCZlos4G1eJkNQkz7ItBubgswQrPrmlhA9a7FsMcrHh1EJC6ZmEmyrjtrhbgtbg5mBPkuvy1N/E6RpEMNw4eOqub8GUdL+I5Gu2jAGQav0N1j4xtTjB9jRMViOJA6G31GhM+IbDuo8LnuQ2YLNqC0dOAM5U78qjxQ5FWZ9YjDDEl0DTUCefwVZWmDQ4tGxt4ls167AczTWSezjSeNwpjDdqjxctuO48qnqmz2p4jROsUwxpHkqXdsylZ0JlExWbQwX82anN9J/VWFqEfzxi4PsoOog2bDwo1OHadErGb1z7xg2EfePZr948f30FlC+NdeXeBtTNuf29/ePUiH9zgE0mVUpMCw4NE5mE47TqzAXOhuqaLEcjQoYTZnGkE5lgP1hmNdk9bDzdWXK8RooJgvqH1zFq3KAnkiHugzknzVKQL5Bm58Q6BPMgVu8PYP2cvThw/p/s9UcM/h69N/5zT0/d7vn/M9f9HP99iuehrbaDkU8ImsvPAjGrHfyZCBUe5s78cjvDVD/uE2glGtbWQCtEMgdj5lcLkbYkoGznd6XHfZ3MBmU2N29oP2v9Bd5dWxvheUvu42aIZaDBGu+hqLT7Mnx2wnVuVhZ+AH7bmOtD6a71R8aKIyQf6eCQ+2bI3QagYw/Ck5Lix5mfHZuNGx9HxP6rQ23u41EfHO18v25urAA2Pcw8/m+JdDD++dgQxJpI3+xxzt/V1D/YdD5tM0dPBNRuoZ3/OCElHORZACFDhmfzJiEgCudoeZ6d1HxK/aDMZ5HLtzYeH218oWh3AypoJ/gbv1QVtD27RbIGuFPGC/1qZORrFlUQXzwIke3j+KIF0T4jA/09QZRPqBWxEuao4mzdHBSLqp0ilFhoAGJq3uqmuxs4NfHIzYw+vsnq6n+RgX8nPu4FsPfp/7PBsKv/Tz7r/0/z7Ot/c8nInLgw/kBjAcBPJQ1q1+sTluvfzb2kfIZ3pWBd4PoVt36yxuVNdwptSpaJEverbgFLFdfQfUMExfV2abOf9BFkWuI2/f3ijujbw69fu79rNUCxhu6jUQI6WqNO+HxyrNzjqaNyboYAGh3KvJ2yZk1fsidjCt+OccsMgCD2wUDItZygWHMLGqFrj+P+CUHnNWBV0ZweC222wO1dQkcbdgfw6W01PaoKxpTt7JO3djRxu7IIrc5uea0V9Qnv3Up8zifBtFJFZKeLcBJPScvhQKt3vMfxfQuULOdDoNQJMrB9zOPSah2G+Gn3mAvuddKOth8EY63o+CUi+8mqompzsKeStHFtEkBKLg8k0HAuIIL5nVqOd1XnOZ3/szWEpLzOHgQeKdzHiZPewwlmmRp9T2gpaZArmmmSYH0ss6aWzhq07fkDnXSH1XTivbMT72AnH2YquvrfHM/6qDrmffs/3ulZff/75PQE9b8H1Z/1/yd4cEOgPqiHyQ5w4Rz21s/iCDPyL67/E7yYZJ1mcnRU+ApTrLMYtFlzN5dLMAz+PKGzYCg/qFeVi39oKxQee08Asc5egKzTVgidc6YtJuvoRZ5JTHLGt6J1Tw8zJjgk6pPRboIeocOuweNUuPUg2OXzm9cvL256CWZcGc9ylgh/BTGL5RzRIRWCahI6ImQ5XgeebyHEKeNCSdryxGsFdNhRH0WEsTlZcc8CP1iKnY0jvB9s75rLAnx+eIV/qObbxvM3H1esKCxxgtz8kkDA0g/kmPedM8zluif6szpPM6xUPN3Qgq4EHk9v2qMaf+85GwJx5HIdReCgF1pV09UhNDFQjTEEnvajsehjbBkUU5wEnmtT5eqqPV2vMFM1LiYjPFIPBh9e/dEcD0WBpMGP+Whauf6rKE/CPSqsbLw4HXZnjVuYGsZIwaqHYCPCi4faqQ10iAQQG2UKlDm4h5tSmGh22IXAi+TSCFfFjDrd1qEuntGSqrRZQ+bfTedguc5WdK6pmMSj0WratfMra7pPTySbbjnBKRmqmhKORZiotdhf2Wq6b3I0OyYb3ZWtHQYBki1wK4drNGR1YrI76E5oyu2i1nnCDdYHU6t7WFUCnbQumQXytrrzhBFcETq4oYvFuzAOEnmGzqOQfHLcaJ1jmx3X+uHYJCT1EeWdgLBABbXAAYWRs8CL5fMtb+Npnz/L77D1Any2GVH1uBRBXobyuOW9URfg4U04ocY24PdUL0iJ6d5ZM1w2WI85dXeXe4SvuQmNOiYkXWXukoEmTKRDlvgcPkDYGkl/Aw6u1MopgkhaAqVD0mZdLTZiK7GVqIiVx+occjd7DfN5131o0n9Al5aU28xfL1J427vpou920KDwyhMRJEKCmEWDqtPBDx4d/1Jj/vXpY5zNceccZvPg8WNVor9D++iRBSIBPASKI+rDC7FzwcqMGDWVfvvgU4A0nQ8+BXivlz6Ns2kOooK3NF1tC+uDgPc6HLHpxTMvxJ3WO5S3cww7s8FUU6No8Itb7BtMdtPdItqJj5K4YHhVfvcO/A6p/c2OxsIvVOa4PbvLEz0FDYyIORb4G8tuCjwq+LqZZVWTEiithQY89erCh2bZTAUPK2cSb8/+K1YK/9uPd09L6hxeM7HWqW88E2wG0qns0b40fDwwdYUHwyUmFanAvNewB4/wgsr/TYd38akOy0vKIS3W/9Pe1e+2bSTx//kUBHM5kYlEW/JHUvkUwI3dooCTBrF7xUHxybRI24r1dSTl1HVd3DvcG96T3PxmdpdLmrIbIHXvDiIQR1zuzn7vzszO/Ha2EIOS32wxOoS7biBnHf4S69EhXHmLOLUTcIl16dCPtXh2x5i0ls4SA9MhTEytYi6xNR2ytamOtjq4+WJPVf4bz4hH/rLHPw/Jf53O9vod/d/mi5X89xhPPdzQEnihbHaWD1hfNJ6dj8ALUNvRhgiP2yGMFpJ51msnrS3xgx+cngsiQyGdSLoQhET1tePOLthXfzZNBhczWstsigEODSYwf4FbEZ/RsL+EXghkXx6xSwCiaEfLHd4tsOr6c1qjZxeir0dM+hYn4PvnRRBwFi6KV+OMwcp5ofQcdeNzJaSWN5una9Mu5sf9dveYdzCpvShlYq3OMiwfw7kc0Up3QLPNN41eRb85gG/6vq+cqDlpS3zfgbj13N1DT/BXvI0jOL9QfGBWBaG7x7BWo8x99dXWU/c0Gl6CFZ7GsjxnMyHoamd6iMmQj9kP+16MHN7ePvXEgkvsrELY5lABesR8KOn2k212yDk+gCigDSMGpwuAdvle/MnTI1LtLSrrQDu3o8qMbVGh9Elhq7BXFBcG7VA4rZU3LI6w1MN7tsgVwI5Vl2HCY1ZAAeDKN6M9ytbtqiTWu7Sz8vD9ZHkij5hYMbXuJ1NSIGvPRMul2K66boxS3VUgNN62bltpvwtCyyoo8U3B9KuqXmrVTM2MIRypuJbPi157Bp8Y5l2GSW+YsEOMr9SzPfwxIcxhpOp1tfF/6efO/s8qq8fd/zdebFT9P7bWN1f7/2M8wH9MUta8Qh/Eptm8A2eub4ARaeZiU2ll+fU4ETjEwGUVJ8y4U2WW5jhvvz/a77p/i8ZJ62t2B3gDOmuuvBwhbutdOjtP4WtA4ht/OTp04xlrmLILElYrmIyZO0sdEZhwEGkBMmYhDM6veeOKuQqxCytHKZhf2NkFpVJSIgAezcYx73k27Xk6+wkok1BccD6nI+IHIHQq+DlahNJR5Ppb60/lGyAo0wS+or+627TLCjET2nQ6WzrmaGpibq6bmDqU9uvDPMqpFy5otQZa1Gg4ysfQRSsFJdi0HZftOImLoJ4aUpG4lErLnPxE9XV9tL5rIhigyUzM9M4EmXMSOkucIRyQHLw+2D083D+EHsd7/R56t3f893CPf+95x87e/sHRbilixqcAiCCnZfglR1rReeIdQ5AHuMyUOI8LqMio+wuuRBsNFbIz5MUuZOBpzJKvkg3toAB+CsWrOlHWqgemAOs/lRI/DWtnXJMnCsTMcscgjm9bqT7iPpH6s/vrhN1526jDNPmkvRfyUZYp6w6K2OaIHUTsMAYZfyWWN8tLulttA3tHH3BF9WXGtuledczPYTrIeciqENgQE9O1RFs2Twds3m9HhhP8PK6Gt8PNdW5Aqq5hOdd7r98TP9d7R387vcM94qd77/YKJ4wzKhngCUyhWAt1xc3cre78xuCJvv+lVxNhQxE0busCdgbeg0LXKJ2mkIKAqVuVjhXtVa+o6p3sSjZjK1YCj9730+Qfi1GaQOWWhflPX3Sre9D/c/vFav//gx4WhV71OmHb4W3AYcNPRznDOcbdyplfX0eTsSOHnau58//y6Pn/fn93781+OIl/hzwe4v/b61tm/tPMB/7/5ouV/++jPE/ch+DfXf9NeJQML8Aef0yGufvvf/7L3SUO+tJ9H310dwPHefLEfb+YiicNjmXYfxjnl047dL9NFDQUm5j4y2QBRi9ugjmdilSgflekiZZdxB337ddEGIek8EdNRzjAD0KnU3Ivdn1xKw7cxXQMhL2qJ3Hofr0YjWP3RB9Dh8Ps6oTZhxPL5sWcMzVxqArjlrzTFNMW8JcnzkZI8a/zC6rNUtsSOws5Xl2D+zLczb7apl9abDlxNklUQbW7hihLaCDSanHKIj2Va3iRwdORXhi7pg1ipxlxoq1WNJmfOFtUS4U2SOIcceqRwOD7IlpQo4wDyqrVYqBCASk8aSJgOmsZ5O2pCWIozhNnm5oa0GBibKGhwMRrvKbcBl7OKnRHl/mlbomNzgkPqtfRVRLlbCqUsXRUkoacltJxspi4ZouGLBTu7b//7q/7eyIPsvDn+nfUHE23IgACoF8LiAH8mn/ILHmK82CCLGcV4pVIZm50FY3GEINCKp1xXZ+QCECynDRqy4yjLgOijXJBCBYcUwhpRhL3s4QPQbmrcFw5k7gymigHg8cj9lu0g5/iAgaqBwR6jBB0ZroALjcS5xfqrgUBw3Qxv5kQpm+G2UN1hJlKtqNP05VNTVaYXOnpk8A1eEjTZ8UN/O8+BvdRLVi/Rx7Y5D/3/p9OZ3X/z2M81f4/zecDwFonaf7FlMAP8H+bW1udSv9vbXdW57+P8igVIG5kADSmmLcOaVtT1+Q0sZcn2LJqtIVV29e74BzOj7tvjwDT2Mjbja7bOGrT/kY8TqPpNsBCqTDaHzmkw+8dFYF5K4SwV7QKxEaOsDf0P3ORjVvtUx7Fg4zvCfJjdVY3ig1QAmMCHPJnABUAKCPJv917/UbCvttDKk50KmajBuGbPdBH7IBO9Lr1erfS88SN2MiUtnNc7wNewn3VdqVwO4yNzLvwGJr2rMB2Ppv+1uJ+MxonbyMoDWEOO4pL9lNcAX3oR4WHNRuRhic0fuIzsXqqmvTBKN3qc4aDSwrsiiJTJmFr01KDUxEoUwFi0a8HGDv+lJJYKAZpEk5wduinDf9D/CEOWsV/N5u3AXUypyjpTNUYDJkgrNMm/Q24C8rPdvFTe0pM1Nm7nRAgB1I6tcL5mNR8pqrKJz1EhbRmgDbhFZhm+hOUu1obKDelzqxSvSEmOsJND10XJo5mGMWWOeTNjPjPKL8IKQYq7M/FhnSOOJatZOMGpbxde/Zs7VkYDycNzNThIs1GV4mYV9za9qwJzJw1aXrD/z4brc7yIBSTavqcJXPbwFWXHrbVbKyWjPstQIHzj47+YSHXT1VfZotT6sm/9z/E4fHz1i+tD/HzP2GeNphUBeCzKUAVWBQEA9GvOCfBx4XBG666qjP6flG64FjQOQXFXZnuWoa/dlRauOzWFmI60zI+OzhjuNQw/mgcF1ikKCfNlHmKsdXgG8T8yUhwsoRg0G00S83XvywnF8hRZT8dx7AEveMVmiZn3JbW+hX3ZXk8Lg02PeTAqF8z4gymeapQVODZwCBLuE4MpWlkrqytLjCNC584zPrGDY3627UbKjv9tWartF44ymYC/OMHt40dDCiYdMcwUp415axpMLus3DZE45K9hYfGap/k5h43jxF6evfmVQMIi1asDJIJoxlTq3Hj0miRxpIpX2nHy+OgOsIkiWwkd45RhHgZnobCMAHOFDqNjeF0L17NfcRfR1kOwnJAZaj88N00f1lOSDK7gq1VW+ZvKnMpkxooq4crdAc8qChQlnTr6rIMMUs/czXyaMRd3obT0Sg8/5kGVxUbiCnNGQVlKBO+uCwIk167I9JnBeQVTcWhAJqgb3BHlZjeOzJdeL/VxuGCaPaq16F5iIbKk6oBsVCjv336Z/wHGsWIbgRh8RIan3i/MZwtpnkjwCFUp2TxT4T+S4XkKv9fVVh9iTwe4P831jer5z/bMANd8f+P8Hie9/kqS/htWFrLfqG2PLbMCoi5JUK4QxHSRdnDPJnM4RshfnJqDLLHi3aWKxyIynEKTz4dE94WzSpkU7MCoug4ERYfXaRwNz1fQLP2Dm/M4UbzMIrjQaQ++J6utlf3kTZPL3DuBEubwGwRpgRQFbuKe+x9tV1Hp2g53DvFDGfPy/JZCqyNRYI8UOx5yMVGwgyFtbfjKGQTxcqW/Ob7PbHKyNvsNkUbB//fwV8WrLxjR/FkyqupLZDyJS8wy48rCnWLaPaJsev77O2StvuTYwsxCtkf7ygTjJ6r47BPC1x39IF9ZMw+uwI9Ly9gx+D/WXiZdhg/0ry23QVzYcDNwqCCNBVNhxez1OwUvM7rgRYeJRgtUXq9xxdKztJrP+CEcXk7HTQFeK+CwYmycxuCE+4Uv8+8mzy+XYvUZubJV6ml+TrRXyucyH2IlvphS5b+cSms1Mp3uYE6MEzun5rteacWBJPqUBv7TlZAi5e9eDq37pVbAsj3WYCF8C7xFYBiUHEOE3MeNuL5rZkaVNBql5SycQx74xRVLAZ5Z+kg75hBbrpfoTgqty/43HGsO9hybK5U9csDehHwy3BvgboJA5ht0VXy8wB8S8quzKgNT38w8qP4dnBz2V3fim/ZoQ4O2D3JvpfrDHomj55y/+vBuY7+V5UXWcdDTvC35JXhcmV2+3/7aP6PhNKz0fkfrP83/F9ne3Nrpf9/jEf3vz6j/T3yeID/f9HeXq/gf3RebKzwPx7lKfj/B07pt+wN2M/MmT9vG59noMAWBvZdY7/IyTgEiZKxwbEKYVuDetHiYwZQeTCGTe3EdgfLsOpVrzkvaAsOFCjV4eI0S/KyrMGwUSayfYtrOZ74TOqIJeeuckQBV9MRNT7WciHIggZolrFBmjZERplA0XmawSxd2vOQJFQr1GgKS4SUWsGG+75eSJLRUCsmba3XUzutj96ppT9OdWRlQ62jbyStzSX0o6xebmtv12aBseEVsSr36NZmURraS8W92nQ8AT6n8ae1ddl8+XkyKwsINQkg/Vp1TxfTEPOwvvgYp/Wdt6SrkWBJ/Dr6JE18TsvAumUp9TpZW1YPkn1pARggNXhzQXIp4APscCdOIFN5w0UceYwMKVeG0ms4ygbGRMhXoF3ecL7wHEeAQfVMU0ApOOAoz1/KZVqRDxQgjimWAf5WhRMfWsHwxwFgnClm/1JB+oQviIz+suN+1MEvt6xwEWdSgMxAgYOC8ZpJH5vIst+9xDFcJfCy+7Em9COgfgo5y8JW2oHIwcc6S0A/DKpICU4kCnm5MWAiIjBnWOlKWEIjlCAKeUzqH4EoIjhMeqQCw0SJbOGIhCxulKugdKiVzfXp0m3T9Yy6ea4hhbyaFvRt8BdVB6DS9j3eSqFDYHVO010akTYR7/i+CNikPVxl6xy45iJ4nBp3rQ0QJ4RReJrh1jZGJuqJym4ACzHcgk6zQ7bDXuU+c76jnJLC3aly8TifqwEUkoNLV5CbT0WYupE84m03UIiMgaMEUpqT0YSK/DTsnL1xFYZk72nmuU+JG1lM/DmgEpOxbx2iCrYkp0xyXHPIMP3tZJsxKKlF4NFFFSq7Y5usd9zZHJ9l+s5wb0S4G0eTH/27hNF641S7wQ7iZBhdwxN+M3Ay9EuFzDgdcDDuBAlfz4j9SXan0yQCkO7Be58igaBsk6AQjflUVkjQghfiugu+wCL1kykWE0x/+sDuOLz+6AUocHBYD/R9Pu7HTY990fwl8wKtyGRmo3LyGOQlEH5bfIyOP7IosmuROXll8M0DH4Pb8vjHAwwWzJgr3aylSXNq5kxFbyc1XeSzIc6WKF3TfaiiZU0YmDIwTWnO5/nU034JBtRgfwJvs1AvUdOzY5gCMt2RO9zTkP/zQTQI4bXGHuLW5zyZo9uKkMWcjzhVWz3vcYGUG7MsThgAkhBqOCpkFTD1wMdCITBLMmQ5IXrSAu/Mfe66XjJvch495LfGK5R0iBAgCs+eTUpanjMvmd8k8+76RnzLSd0b0MYxf7+B98ZxN9w4u8Wlvu7NpN/QaJ0I71D4WbsI1tCc+ptn9caZx/7f52sZx7dQOlXsNRMsDoOaiH9jjbpWvt4N189us0AjYvP9Uj13Im7opggeLnygQCufUpDk4ZlLI4TOK7aIMXYxHKh3f6jD1KxnQ2SBCuNZT7xQmCZz3N7ie8IG0fIfznMgWhnrKuzu2iiDNfVKb780aVfNQWx/AytPKQ9vivcTCJwztuKvG1Z5Uh5WO1rtd7R/eEQkOGXggGAYkzQlwwwMUe8qwv4SyHoyS697+F9B+EmyJs0hGnvqWML75LFtTgzEsPZKkbh6Vs/qWT2rZ/WsntWzelbPH/n8B5aigp8AoAAA"
shutil.rmtree('/content/longnet', ignore_errors=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(B64)), mode='r:gz') as t: t.extractall('/content')
assert os.path.exists('/content/longnet/train.py')
os.chdir('/content/longnet'); sys.path.insert(0, '/content/longnet'); sys.path.insert(0, '/content/longnet/scripts')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nibabel', 'SimpleITK', 'pyyaml', 'pandas', 'scipy'])
print('ready in', os.getcwd(), os.listdir('.'))

/tmp/ipykernel_6970/3661664751.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(fileobj=io.BytesIO(base64.b64decode(B64)), mode='r:gz') as t: t.extractall('/content')


ready in /content/longnet ['configs', 'train.py', 'scripts', 'requirements.txt', 'README.md', 'longnet']


In [21]:
# Upload longnet.tar.gz (the file Claude gave you) into WORK on Drive once, or upload here:
if not os.path.exists(f'{WORK}/longnet.tar.gz'):
    from google.colab import files
    up = files.upload()                     # choose longnet.tar.gz
    open(f'{WORK}/longnet.tar.gz','wb').write(next(iter(up.values())))
!cd /content && tar xzf {WORK}/longnet.tar.gz && cd longnet && pip install -q nibabel SimpleITK pyyaml pandas scipy tcia_utils
%cd /content/longnet

/content/longnet


## 1. Smoke test (≈1 min on GPU) – confirms code runs before touching real data

In [32]:
!python train.py --synthetic --epochs 5 --base 8 --n 32 --size 32 --out /content/smoke.json | tail -3

ep003 loss 2.498 acc 0.00 f1 0.00 dice g/s 0.17/0.33 (0s)
ep004 loss 2.444 acc 0.00 f1 0.00 dice g/s 0.17/0.33 (0s)
TEST {'rano_acc': 0.2, 'rano_macro_f1': 0.16666666666666666, 'dice_growth': 0.0860325346842706, 'dice_shrink': 0.420445039452646, 'hd95_growth': 17.42645517458605, 'hd95_shrink': 2.0214983674153264}


In [22]:
!python train.py --synthetic --epochs 5 --base 8 --n 32 --size 32 --out /content/smoke.json | tail -3

ep003 loss 2.498 acc 0.00 f1 0.00 dice g/s 0.17/0.33 (1s)
ep004 loss 2.444 acc 0.00 f1 0.00 dice g/s 0.17/0.33 (1s)
TEST {'rano_acc': 0.2, 'rano_macro_f1': 0.16666666666666666, 'dice_growth': 0.08603367991020214, 'dice_shrink': 0.420445039452646, 'hd95_growth': 17.42645517458605, 'hd95_shrink': 2.0214983674153264}


## 2. Data
Start with **Brain-Tumor-Progression** (20 GBM patients, small) to debug the pipeline, then add **Yale-Brain-Mets-Longitudinal**.
Download from TCIA (https://www.cancerimagingarchive.net, collection pages → *Download* → NBIA Data Retriever, or `tcia_utils` below).
Some TCIA collections are DICOM; convert to NIfTI (`dcm2niix`) and keep tumor masks as NIfTI. Check each collection's license/terms.
**BraTS** is mostly single-timepoint → use for encoder pretraining only, not for pairs.

In [2]:
# Option A: tcia_utils (public collections). Collection name must match the TCIA page exactly.
from tcia_utils import nbia
COLLECTION = 'UPENN-GBM'
series = nbia.getSeries(collection=COLLECTION)

if series is None:
    print(f"Error: Collection '{COLLECTION}' returned no data. Check spelling or credentials.")
else:
    print(len(series), 'series')
    import pandas as pd
    df = pd.DataFrame(series)
visits = df.groupby('PatientID').StudyDate.nunique()
print('patients:', len(visits), '| with >=2 scan dates:', (visits >= 2).sum())
print(visits.value_counts().sort_index())

# the preprocessed series (skull-stripped, co-registered) are the useful ones
proc = df[df.SeriesDescription.str.contains('Processed_CaPTk', na=False)]
print('processed series:', len(proc), 'of', len(df))
print(proc.SeriesDescription.str.replace(': Processed_CaPTk', '', regex=False).str.strip().value_counts().head(25))

3680 series
patients: 630 | with >=2 scan dates: 43
StudyDate
1    587
2     42
3      1
Name: count, dtype: int64
processed series: 2607 of 3680
SeriesDescription
Axial T2 tse                  402
t2_Flair_axial                378
t1 axial stealth-post         371
t1 axial                      338
T2 SAG SPACE                   97
AX T2                          72
t1 axial stealth-post_TERA     66
T2 AXIAL                       65
t2_Flair_axial_TERA            64
t1 axial stealth-post_T        64
t2_Flair_axial_T               63
t1 axial_T                     63
AX T1 PRE                      55
T2 AXIAL FLAIR                 44
AX T1 3D POST STEALTH          42
t1 axial_TERA                  41
AX FLAIR                       35
AX T1 3D PRE                   31
AX T1 MPRAGE ISOTROPIC         30
t1 axial_ 3D_TERA              25
T1 MPRAGE AXIAL IPAT           19
AXIAL FLAIR                    18
t1 axial_ 3D                   13
AXIAL T2 FLAIR                 10
AX T2 FLAIR         

In [7]:
import os, collections

# Fallback definitions in case the kernel restarted
if 'WORK' not in globals():
    WORK = '/content/drive/MyDrive/mtech_longnet'
if 'COLLECTION' not in globals():
    COLLECTION = 'UPENN-GBM'

# Dynamically target the raw download folder on Google Drive
root = f"{WORK}/raw/{COLLECTION}"
cnt = collections.Counter(); ex = {}

if not os.path.exists(root):
    print(f"Warning: The dataset path '{root}' does not exist yet. Please execute the download cell (d3114eab) first.")
else:
    for d, _, fs in os.walk(root):
        for f in fs:
            ext = '.nii.gz' if f.endswith('.nii.gz') else os.path.splitext(f)[1].lower()
            cnt[ext] += 1; ex.setdefault(ext, os.path.join(d, f))
    print('file types:', dict(cnt))
    for e, p in ex.items(): print(e, '->', os.path.relpath(p, root))
    print('top level:', os.listdir(root)[:10])
    total = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(root) for f in fs)
    print('size GB:', round(total / 1e9, 1))

file types: {'': 77, '.dcm': 26267}
 -> 1.3.6.1.4.1.14519.5.2.1.182634420755264781924968938417569010478/LICENSE
.dcm -> 1.3.6.1.4.1.14519.5.2.1.182634420755264781924968938417569010478/1-01.dcm
top level: ['1.3.6.1.4.1.14519.5.2.1.182634420755264781924968938417569010478', '1.3.6.1.4.1.14519.5.2.1.213205905013069219671876086260062935951', '1.3.6.1.4.1.14519.5.2.1.326941089799128875716457447813731935085', '1.3.6.1.4.1.14519.5.2.1.228154008480265056687138530279461887340', '1.3.6.1.4.1.14519.5.2.1.76934004063291864081547030012238457331', '1.3.6.1.4.1.14519.5.2.1.203487181661673601689905265504325677572', '1.3.6.1.4.1.14519.5.2.1.269607322454961545178354587485557855557', '1.3.6.1.4.1.14519.5.2.1.275026661744302022599791902247423457782', '1.3.6.1.4.1.14519.5.2.1.170870999839860708966191655293902879313', '1.3.6.1.4.1.14519.5.2.1.23166033272237112806900422604932457206']
size GB: 2.7


In [9]:
import zipfile, os, collections

# Fallback definition in case the kernel restarted
if 'WORK' not in globals():
    WORK = '/content/drive/MyDrive/mtech_longnet'

Z = f'{WORK}/archive.zip'      # pointing to your persistent Drive folder

if not os.path.exists(Z):
    print(f"Error: Archive file not found at '{Z}'.")
    print("Please verify if you have uploaded 'archive.zip' to your Google Drive workspace, or update the 'Z' path.")
else:
    print('size GB:', round(os.path.getsize(Z)/1e9, 2))
    with zipfile.ZipFile(Z) as z:
        names = z.namelist(); print(len(names), 'files')
        cnt = collections.Counter('.nii.gz' if n.endswith('.nii.gz') else os.path.splitext(n)[1].lower() for n in names)
        print(dict(cnt)); print('\n'.join(names[:25]))

Error: Archive file not found at '/content/drive/MyDrive/mtech_longnet/archive.zip'.
Please verify if you have uploaded 'archive.zip' to your Google Drive workspace, or update the 'Z' path.


In [17]:
cols = nbia.getCollections()
names = sorted(c['Collection'] if isinstance(c, dict) else str(c) for c in cols)
print(len(names), 'collections total')
for n in names:
    if any(k in n.lower() for k in ['yale','brain','glio','gbm','progress','mets','longitud','brats','ucsf','rhuh']):
        print(n)

156 collections total
GBM-DSC-MRI-DRO
ICDC-Glioma
Spine-Mets-CT-SEG
UPENN-GBM


In [18]:
import subprocess, os, zipfile, collections
# 1) locate it anywhere in your Drive (My Drive; also shortcuts)
hits = subprocess.run("find /content/drive -iname 'archive*.zip' 2>/dev/null | head", shell=True, capture_output=True, text=True).stdout.split()
print(hits)

['/content/drive/MyDrive/mtech_longnet/archive.zip']


In [25]:
import zipfile, os, re, collections, glob
Z = '/content/drive/MyDrive/mtech_longnet/archive.zip'
OUT = '/content/btp'
if not os.path.exists(OUT):
    with zipfile.ZipFile(Z) as z: z.extractall(OUT)

series_dirs = sorted({os.path.dirname(p) for p in glob.glob(f'{OUT}/**/*.dcm', recursive=True)})
def parse(d):
    parts = d.replace(OUT + '/', '').split('/')          # Brain-Tumor-Progression / PGBM-001 / study / series
    return parts[1], parts[2], parts[3]
rows = [parse(d) for d in series_dirs]
names = collections.Counter(re.sub(r'^[\d.]+-|-\d+$', '', s) for _, _, s in rows)
print(len(rows), 'series,', len({r[0] for r in rows}), 'patients')
print('\nseries names (count):'); [print(f'  {n}: {c}') for n, c in names.most_common()]
studies = collections.defaultdict(set)
for p, st, _ in rows: studies[p].add(st)
print('\nstudies per patient:', collections.Counter(len(v) for v in studies.values()))
print('example patient:'); [print(' ', r[1], '|', r[2]) for r in rows[:25]]

362 series, 19 patients

series names (count):
  T1post: 38
  T1prereg: 38
  FLAIRreg: 38
  ADCreg: 38
  T2reg: 38
  MaskTumor: 37
  dT1: 36
  sRCBVreg: 33
  nRCBVreg: 33
  nCBFreg: 33

studies per patient: Counter({2: 19})
example patient:
  04-02-1992-FH-HEADBrain Protocols-79896 | 11.000000-T1post-80644
  04-02-1992-FH-HEADBrain Protocols-79896 | 29904.000000-dT1-04743
  04-02-1992-FH-HEADBrain Protocols-79896 | 34907.000000-T1prereg-43695
  04-02-1992-FH-HEADBrain Protocols-79896 | 35623.000000-FLAIRreg-79237
  04-02-1992-FH-HEADBrain Protocols-79896 | 35624.000000-ADCreg-47789
  04-02-1992-FH-HEADBrain Protocols-79896 | 35628.000000-sRCBVreg-27603
  04-02-1992-FH-HEADBrain Protocols-79896 | 35629.000000-nRCBVreg-30349
  04-02-1992-FH-HEADBrain Protocols-79896 | 35630.000000-nCBFreg-63461
  04-02-1992-FH-HEADBrain Protocols-79896 | 37908.000000-T2reg-21119
  04-02-1992-FH-HEADBrain Protocols-79896 | 5388.000000-MaskTumor-63335
  11-19-1991-FH-HEADBrain Protocols-40993 | 10008.00000

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [23]:
import os, zipfile, collections
Z = '/content/drive/MyDrive/mtech_longnet/archive.zip'
print('exists:', os.path.exists(Z))
if os.path.exists(Z):
    print('size GB:', round(os.path.getsize(Z)/1e9, 2))
    with zipfile.ZipFile(Z) as z:
        names = z.namelist(); print(len(names), 'files')
        cnt = collections.Counter('.nii.gz' if n.endswith('.nii.gz') else os.path.splitext(n)[1].lower() for n in names)
        print(dict(cnt)); print('\n'.join(names[:30]))
else:
    print([os.path.join(d,f) for d,_,fs in os.walk('/content/drive/MyDrive') for f in fs if f.lower().endswith('.zip')][:20])

exists: True
size GB: 1.3
8798 files
{'.dcm': 8798}
Brain-Tumor-Progression/PGBM-001/04-02-1992-FH-HEADBrain Protocols-79896/11.000000-T1post-80644/1-01.dcm
Brain-Tumor-Progression/PGBM-001/04-02-1992-FH-HEADBrain Protocols-79896/11.000000-T1post-80644/1-02.dcm
Brain-Tumor-Progression/PGBM-001/04-02-1992-FH-HEADBrain Protocols-79896/11.000000-T1post-80644/1-03.dcm
Brain-Tumor-Progression/PGBM-001/04-02-1992-FH-HEADBrain Protocols-79896/11.000000-T1post-80644/1-04.dcm
Brain-Tumor-Progression/PGBM-001/04-02-1992-FH-HEADBrain Protocols-79896/11.000000-T1post-80644/1-05.dcm
Brain-Tumor-Progression/PGBM-001/04-02-1992-FH-HEADBrain Protocols-79896/11.000000-T1post-80644/1-06.dcm
Brain-Tumor-Progression/PGBM-001/04-02-1992-FH-HEADBrain Protocols-79896/11.000000-T1post-80644/1-07.dcm
Brain-Tumor-Progression/PGBM-001/04-02-1992-FH-HEADBrain Protocols-79896/11.000000-T1post-80644/1-08.dcm
Brain-Tumor-Progression/PGBM-001/04-02-1992-FH-HEADBrain Protocols-79896/11.000000-T1post-80644/1-09.dcm
Bra

In [13]:
import zipfile, os, collections

# Fallback definition in case the kernel restarted
if 'WORK' not in globals():
    WORK = '/content/drive/MyDrive/mtech_longnet'

Z = f'{WORK}/archive.zip'      # pointing to your persistent Drive folder

if not os.path.exists(Z):
    print(f"Error: Archive file not found at '{Z}'.")
    print("Please verify if you have uploaded 'archive.zip' to your Google Drive workspace, or update the 'Z' path.")
else:
    print('size GB:', round(os.path.getsize(Z)/1e9, 2))
    with zipfile.ZipFile(Z) as z:
        names = z.namelist(); print(len(names), 'files')
        cnt = collections.Counter('.nii.gz' if n.endswith('.nii.gz') else os.path.splitext(n)[1].lower() for n in names)
        print(dict(cnt)); print('\n'.join(names[:25]))

Error: Archive file not found at '/content/drive/MyDrive/mtech_longnet/archive.zip'.
Please verify if you have uploaded 'archive.zip' to your Google Drive workspace, or update the 'Z' path.


In [33]:
import zipfile
Z = f'{WORK}/archive.zip'
RAWDIR = '/content/btp'
if not os.path.exists(RAWDIR):
    with zipfile.ZipFile(Z) as z: z.extractall(RAWDIR)
print(len(os.listdir(RAWDIR)), 'top-level entries')

1 top-level entries


In [36]:
import btp_convert, pandas as pd
NIFTI = f'{WORK}/nifti'
man = btp_convert.convert(RAWDIR, NIFTI)          # a few minutes
man.to_csv(f'{WORK}/manifest.csv', index=False)
print(len(man), 'complete timepoints |', man.patient_id.nunique(), 'patients')
man[['patient_id','timepoint']].head(10)

skip (missing series): PGBM-019 10-02-1991-FH-HEADBrain Protocols-94010 ['mask']
36 complete timepoints | 18 patients


,patient_id,timepoint
0,PGBM-001,1992-04-02
1,PGBM-001,1991-11-19
2,PGBM-002,1997-01-17
3,PGBM-002,1996-08-13
4,PGBM-003,1995-03-29
5,PGBM-003,1995-10-17
6,PGBM-004,1994-01-12
7,PGBM-004,1993-06-22
8,PGBM-005,1992-05-05
9,PGBM-005,1991-07-02


### Manifest
`prepare_pairs.py` needs a CSV: `patient_id,timepoint,t1,t1ce,t2,flair,mask`.
Folder layout differs between collections, so **edit the matching rule in the next cell** after looking at your files.
Timepoint must sort chronologically (study date, or an integer visit index).

In [28]:
import glob, re, os, pandas as pd
NIFTI = f'{WORK}/nifti'          # <- where your converted/organised NIfTI files are
def find(pid, tp, key):		  # EDIT: adapt pattern to your naming
    hits = glob.glob(f'{NIFTI}/{pid}/{tp}/*{key}*.nii*')
    return hits[0] if hits else None
rows = []
for pdir in sorted(glob.glob(f'{NIFTI}/*')):
    pid = os.path.basename(pdir)
    for tp in sorted(os.listdir(pdir)):
        r = dict(patient_id=pid, timepoint=tp, t1=find(pid,tp,'T1pre'), t1ce=find(pid,tp,'T1post'),
                 t2=find(pid,tp,'T2'), flair=find(pid,tp,'FLAIR'), mask=find(pid,tp,'seg'))
        if all(r[k] for k in ['t1','t1ce','t2','flair','mask']): rows.append(r)

if not rows:
    print(f"Warning: No complete timepoints found in '{NIFTI}'.")
    print("Please make sure you have converted your DICOM files to NIfTI in {WORK}/nifti before running this step.")
    man = pd.DataFrame(columns=['patient_id', 'timepoint', 't1', 't1ce', 't2', 'flair', 'mask'])
else:
    man = pd.DataFrame(rows); man.to_csv(f'{WORK}/manifest.csv', index=False)
    print(len(man), 'complete timepoints,', man.patient_id.nunique(), 'patients'); man.head()

Please make sure you have converted your DICOM files to NIfTI in {WORK}/nifti before running this step.


## 3. Preprocess → cached pairs
Skull-strip first if your scans are not stripped (HD-BET: `pip install hd-bet`; BraTS is already stripped).
`--register` rigidly aligns timepoint 2 → timepoint 1 (T1ce anchor) and carries the mask along.

In [39]:
!PYTHONPATH=/content/longnet python scripts/prepare_pairs.py {WORK}/manifest.csv {WORK}/cache64 --size 64 --register
!ls {WORK}/cache64 | wc -l

saved PGBM-001 0
saved PGBM-002 1
saved PGBM-003 2
saved PGBM-004 3
saved PGBM-005 4
saved PGBM-006 5
saved PGBM-007 6
saved PGBM-008 7
saved PGBM-009 8
saved PGBM-010 9
saved PGBM-011 10
saved PGBM-012 11
saved PGBM-013 12
saved PGBM-014 13
saved PGBM-015 14
saved PGBM-016 15
saved PGBM-017 16
saved PGBM-018 17
18


In [40]:
# Sanity check on label distribution (derived from masks - see longnet/labels.py)
import glob, numpy as np
from longnet.labels import rano_from_volumes, RANO_CLASSES
vols=[]
for f in sorted(glob.glob(f'{WORK}/cache64/*.npz')):
    z=np.load(f); v1,v2=z['mask1'].sum(),z['mask2'].sum(); vols.append((str(z['pid']),int(v1),int(v2),RANO_CLASSES[rano_from_volumes(v1,v2)]))
print(pd.DataFrame(vols,columns=['patient','vol_t1','vol_t2','derived_RANO']))

     patient  vol_t1  vol_t2 derived_RANO
0   PGBM-001     339     819           PD
1   PGBM-002    1560    1797           SD
2   PGBM-003     267     129           SD
3   PGBM-004     759     723           SD
4   PGBM-005     198      48           PR
5   PGBM-006     418     573           SD
6   PGBM-007     396     468           SD
7   PGBM-008     602      84           PR
8   PGBM-009      30      15           SD
9   PGBM-010     471     471           SD
10  PGBM-011     177     228           SD
11  PGBM-012     522    1324           PD
12  PGBM-013    1565    1897           SD
13  PGBM-014     456     332           SD
14  PGBM-015      57     210           PD
15  PGBM-016     750     954           SD
16  PGBM-017    2701    3333           SD
17  PGBM-018    1545    1701           SD


## 4. Train + baselines/ablations
Same seeds and splits for every run. Use ≥3 seeds for the final table (edit `SEEDS`).

In [ ]:
VARIANTS = {'full': '', 'single_tp': '--mode single'}      # add: 'no_cross_attn': '--no-cross-attn', 'no_weight_share': '--no-share'
K, EP = 5, 100
os.makedirs(f'{WORK}/runs', exist_ok=True)
for name, flags in VARIANTS.items():
    for f in range(K):
        !python train.py --cache {WORK}/cache64 --folds {K} --fold {f} --epochs {EP} --base 16 --bs 2 --amp {flags} --out {WORK}/runs/{name}_f{f}.json | grep -E "patients per|TEST"


{'test': 4, 'val': 4, 'train': 10} patients per split
TEST {'rano_acc': 1.0, 'rano_macro_f1': 1.0, 'dice_growth': 0.2224784369374956, 'dice_shrink': 0.0, 'hd95_growth': 9.7316434703062, 'hd95_shrink': nan}
{'test': 4, 'val': 4, 'train': 10} patients per split
TEST {'rano_acc': 0.5, 'rano_macro_f1': 0.2222222222222222, 'dice_growth': 0.13800592998235514, 'dice_shrink': 0.0, 'hd95_growth': 15.229272926783493, 'hd95_shrink': nan}
{'test': 4, 'val': 3, 'train': 11} patients per split
TEST {'rano_acc': 0.75, 'rano_macro_f1': 0.42857142857142855, 'dice_growth': 0.16984631850895, 'dice_shrink': 0.0, 'hd95_growth': 19.788193941809833, 'hd95_shrink': nan}
{'test': 3, 'val': 3, 'train': 12} patients per split
TEST {'rano_acc': 0.0, 'rano_macro_f1': 0.0, 'dice_growth': 0.2832141924142885, 'dice_shrink': 0.1096410694724846, 'hd95_growth': 6.544200945962641, 'hd95_shrink': 7.0030845240544695}


ERROR:tcia_utils.nbia:Exception during download for series 1.3.6.1.4.1.14519.5.2.1.253414618989570099999160046512911577184: HTTPSConnectionPool(host='nbia.cancerimagingarchive.net', port=443): Max retries exceeded with url: /nbia-api/services/v4/getImage?NewFileNames=Yes&SeriesInstanceUID=1.3.6.1.4.1.14519.5.2.1.253414618989570099999160046512911577184 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7ea83125aad0>, 'Connection to nbia.cancerimagingarchive.net timed out. (connect timeout=None)'))


{'test': 3, 'val': 4, 'train': 11} patients per split
TEST {'rano_acc': 0.6666666666666666, 'rano_macro_f1': 0.4, 'dice_growth': 0.1726674762193888, 'dice_shrink': 0.0, 'hd95_growth': 4.1831317843458455, 'hd95_shrink': nan}
/content/longnet/train.py:47: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step(); m = evaluate(model, L(va, False), dev)


ERROR:tcia_utils.nbia:Exception during download for series 1.3.6.1.4.1.14519.5.2.1.153765194881401441708092508611837745335: [Errno 28] No space left on device
ERROR:tcia_utils.nbia:Exception during download for series 1.3.6.1.4.1.14519.5.2.1.167727330421096360000080775020257208043: [Errno 28] No space left on device
ERROR:tcia_utils.nbia:Exception during download for series 1.3.6.1.4.1.14519.5.2.1.106157189843633989874638455691669803918: [Errno 28] No space left on device
ERROR:tcia_utils.nbia:Exception during download for series 1.3.6.1.4.1.14519.5.2.1.159423188447809044284254245533714428713: [Errno 28] No space left on device
ERROR:tcia_utils.nbia:Exception during download for series 1.3.6.1.4.1.14519.5.2.1.6933881882035220155947696628261923896: [Errno 28] No space left on device
ERROR:tcia_utils.nbia:Exception during download for series 1.3.6.1.4.1.14519.5.2.1.61199083580292630272965576348689286816: [Errno 28] No space left on device
ERROR:tcia_utils.nbia:Exception during download f

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SEEDS = [0]          # final report: [0,1,2]
CFG = {
  'full':          '',
  'single_tp':     '--mode single',
  'no_cross_attn': '--no-cross-attn',
  'no_weight_share': '--no-share',
}
EP = 100
os.makedirs(f'{WORK}/runs', exist_ok=True)
for name, flags in CFG.items():
    for s in SEEDS:
        !python train.py --cache {WORK}/cache96 --epochs {EP} --base 16 --bs 2 --amp --seed {s} {flags} --out {WORK}/runs/{name}_s{s}.json


## 5. Results table (mean ± std over seeds) – paste into the report

In [ ]:
import json, glob, numpy as np, pandas as pd
rows=[]
for name in CFG:
    rs=[json.load(open(f))['test'] for f in glob.glob(f'{WORK}/runs/{name}_s*.json')]
    if not rs: continue
    d={'model':name,'n_seeds':len(rs)}
    for k in rs[0]:
        v=np.array([r[k] for r in rs],float); d[k]=f'{np.nanmean(v):.3f} ± {np.nanstd(v):.3f}'
    rows.append(d)
res=pd.DataFrame(rows); res.to_csv(f'{WORK}/results.csv',index=False); res

In [ ]:
# training curves for the full model
import matplotlib.pyplot as plt
h=json.load(open(sorted(glob.glob(f'{WORK}/runs/full_s*.json'))[0]))['history']
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
ax[0].plot([x['loss'] for x in h]); ax[0].set_title('train loss'); ax[0].set_xlabel('epoch')
for k in ['rano_macro_f1','dice_growth','dice_shrink']: ax[1].plot([x[k] for x in h],label=k)
ax[1].legend(); ax[1].set_title('validation'); ax[1].set_xlabel('epoch'); plt.tight_layout()
plt.savefig(f'{WORK}/curves.png',dpi=200)

## 6. Qualitative figures: delta maps + cross-time attention (for slides / explainability)

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from longnet.model import DualStreamNet
from longnet.data import CachedPairs, patient_split
pids=[str(np.load(f)['pid']) for f in glob.glob(f'{WORK}/cache96/*.npz')]
te=CachedPairs(f'{WORK}/cache96', patient_split(pids)['test'])
m=DualStreamNet(base=16).cuda().eval(); m.load_state_dict(torch.load(f'{WORK}/runs/full_s0.pt'))
for i in range(min(4,len(te))):
    b=te[i]
    with torch.no_grad(): o=m(b['t1'][None].cuda(),b['t2'][None].cuda(),keep_attn=True)
    pred=o['delta'].argmax(1)[0].cpu(); z=int(b['mask1'].sum((1,2)).argmax())
    fig,ax=plt.subplots(1,4,figsize=(14,3.5))
    ax[0].imshow(b['t1'][1,z],cmap='gray');ax[0].set_title('t1 (T1ce)')
    ax[1].imshow(b['t2'][1,z],cmap='gray');ax[1].set_title('t2 (T1ce)')
    ax[2].imshow(b['delta'][z],vmin=0,vmax=2,cmap='coolwarm');ax[2].set_title('GT delta (1=growth, 2=shrink)')
    ax[3].imshow(pred[z],vmin=0,vmax=2,cmap='coolwarm');ax[3].set_title(f'pred; RANO pred={o["rano"].argmax().item()} gt={int(b["rano"])}')
    for a in ax: a.axis('off')
    plt.savefig(f'{WORK}/qual_{i}.png',dpi=200,bbox_inches='tight'); plt.show()
# attention weights of the last sample: m.attn.last_attn  -> [1, N_t1, N_t2] over bottleneck tokens (12^3 for 96^3 input)

**Notes for the report**
- Delta maps / RANO grades are derived from masks (`longnet/labels.py`); say so, or use clinical RANO labels where the dataset has them.
- Report mean ± std over ≥3 seeds, patient-level splits (already enforced).
- If RANO classes are imbalanced (CR is rare), also report per-class recall and a confusion matrix.